#  Data Extraction & Validation — Image / Text
> Script cơ bản để trích xuất và kiểm tra chất lượng dữ liệu trước khi đưa vào ML pipeline

---

## 1️ Cài đặt thư viện cần thiết

In [1]:
# Chạy cell này lần đầu để cài thư viện
!pip install pillow pandas numpy matplotlib tqdm langdetect -q

  DEPRECATION: Building 'langdetect' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'langdetect'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\kaigi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2️⃣ Import thư viện

In [ ]:
import os
import json
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter

print('✅ Import thư viện thành công!')

## 3️⃣ Cấu hình đường dẫn dữ liệu

In [ ]:
# ========================================
# ⚙️ THAY ĐỔI CÁC ĐƯỜNG DẪN NÀY CHO PHÙ HỢP
# ========================================

CONFIG = {
    # Thư mục chứa ảnh (để trống nếu không dùng)
    'image_dir': './data/images',

    # File CSV chứa text (để trống nếu không dùng)
    'text_csv_path': './data/texts.csv',

    # Tên cột chứa nội dung text trong CSV
    'text_column': 'text',

    # Tên cột chứa nhãn (label) trong CSV — để trống nếu không có
    'label_column': 'label',

    # Định dạng ảnh hợp lệ
    'valid_image_formats': ['.jpg', '.jpeg', '.png', '.bmp', '.webp'],

    # Kích thước ảnh tối thiểu (pixel)
    'min_image_size': (32, 32),

    # Kích thước file ảnh tối đa (MB)
    'max_image_size_mb': 10,

    # Độ dài text tối thiểu / tối đa (ký tự)
    'min_text_length': 10,
    'max_text_length': 10000,

    # Thư mục lưu báo cáo
    'report_dir': './validation_report'
}

# Tạo thư mục báo cáo
Path(CONFIG['report_dir']).mkdir(parents=True, exist_ok=True)

print('✅ Cấu hình xong!')
print(f"📁 Image dir : {CONFIG['image_dir']}")
print(f"📄 Text CSV  : {CONFIG['text_csv_path']}")
print(f"📊 Report dir: {CONFIG['report_dir']}")

---
## 🖼️ PHẦN A — IMAGE EXTRACTION & VALIDATION

### A1. Trích xuất danh sách ảnh

In [ ]:
def extract_images(image_dir, valid_formats):
    """Quét thư mục và lấy toàn bộ ảnh hợp lệ."""
    image_dir = Path(image_dir)
    if not image_dir.exists():
        print(f'⚠️  Thư mục không tồn tại: {image_dir}')
        return []

    all_files = list(image_dir.rglob('*'))
    image_files = [
        f for f in all_files
        if f.is_file() and f.suffix.lower() in valid_formats
    ]
    print(f'📂 Tổng file quét được : {len(all_files)}')
    print(f'🖼️  Ảnh hợp lệ tìm thấy: {len(image_files)}')
    return image_files


image_files = extract_images(
    CONFIG['image_dir'],
    CONFIG['valid_image_formats']
)

### A2. Validate từng ảnh

In [ ]:
def validate_single_image(filepath, config):
    """Kiểm tra một ảnh và trả về dict kết quả."""
    result = {
        'filepath'   : str(filepath),
        'filename'   : filepath.name,
        'format'     : filepath.suffix.lower(),
        'size_mb'    : round(filepath.stat().st_size / 1e6, 3),
        'width'      : None,
        'height'     : None,
        'mode'       : None,
        'is_corrupt' : False,
        'errors'     : []
    }

    # Kiểm tra dung lượng
    if result['size_mb'] > config['max_image_size_mb']:
        result['errors'].append(f"Quá lớn: {result['size_mb']} MB")

    # Mở ảnh
    try:
        with Image.open(filepath) as img:
            img.verify()  # Phát hiện ảnh corrupt

        with Image.open(filepath) as img:
            result['width']  = img.width
            result['height'] = img.height
            result['mode']   = img.mode

        # Kiểm tra kích thước tối thiểu
        min_w, min_h = config['min_image_size']
        if img.width < min_w or img.height < min_h:
            result['errors'].append(
                f"Quá nhỏ: {img.width}x{img.height} (tối thiểu {min_w}x{min_h})"
            )

    except Exception as e:
        result['is_corrupt'] = True
        result['errors'].append(f'Corrupt: {str(e)}')

    result['is_valid'] = (not result['is_corrupt']) and (len(result['errors']) == 0)
    return result


# Chạy validation
image_results = []
for f in tqdm(image_files, desc='Validating images'):
    image_results.append(validate_single_image(f, CONFIG))

df_images = pd.DataFrame(image_results)

if not df_images.empty:
    valid_count   = df_images['is_valid'].sum()
    invalid_count = (~df_images['is_valid']).sum()
    corrupt_count = df_images['is_corrupt'].sum()
    print(f'\n📊 KẾT QUẢ VALIDATION ẢNH')
    print(f'   ✅ Hợp lệ  : {valid_count}')
    print(f'   ❌ Lỗi     : {invalid_count}')
    print(f'   💀 Corrupt : {corrupt_count}')
else:
    print('⚠️  Không có ảnh nào để validate.')

### A3. Kiểm tra ảnh trùng lặp (duplicate)

In [ ]:
def compute_hash(filepath, chunk_size=8192):
    """Tính MD5 hash của file."""
    h = hashlib.md5()
    try:
        with open(filepath, 'rb') as f:
            while chunk := f.read(chunk_size):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None


if image_files:
    print('🔍 Đang tính hash để tìm ảnh trùng...')
    hashes = [compute_hash(f) for f in tqdm(image_files, desc='Hashing')]

    hash_counts = Counter(h for h in hashes if h)
    duplicates  = {h: c for h, c in hash_counts.items() if c > 1}

    print(f'\n🔁 Số nhóm ảnh trùng lặp: {len(duplicates)}')
    total_dup_files = sum(c - 1 for c in duplicates.values())
    print(f'🗑️  Số ảnh dư thừa cần xóa: {total_dup_files}')
else:
    print('⚠️  Bỏ qua bước kiểm tra duplicate (không có ảnh).')

### A4. Visualize kết quả ảnh

In [ ]:
if not df_images.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Image Validation Report', fontsize=14, fontweight='bold')

    # --- Biểu đồ 1: Valid vs Invalid ---
    counts = [df_images['is_valid'].sum(), (~df_images['is_valid']).sum()]
    axes[0].pie(counts, labels=['Valid', 'Invalid'],
                colors=['#4CAF50', '#F44336'],
                autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Tỉ lệ Valid / Invalid')

    # --- Biểu đồ 2: Phân bố định dạng ảnh ---
    fmt_counts = df_images['format'].value_counts()
    axes[1].bar(fmt_counts.index, fmt_counts.values,
                color='#2196F3', edgecolor='white')
    axes[1].set_title('Phân bố định dạng ảnh')
    axes[1].set_xlabel('Format')
    axes[1].set_ylabel('Số lượng')

    # --- Biểu đồ 3: Phân bố kích thước file ---
    valid_df = df_images[df_images['is_valid']]
    if not valid_df.empty:
        axes[2].hist(valid_df['size_mb'], bins=20,
                     color='#FF9800', edgecolor='white')
        axes[2].set_title('Phân bố kích thước file (MB)')
        axes[2].set_xlabel('Size (MB)')
        axes[2].set_ylabel('Số lượng')
    else:
        axes[2].text(0.5, 0.5, 'Không có dữ liệu',
                     ha='center', va='center', transform=axes[2].transAxes)

    plt.tight_layout()
    report_path = f"{CONFIG['report_dir']}/image_validation.png"
    plt.savefig(report_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Đã lưu biểu đồ: {report_path}')
else:
    print('⚠️  Không có dữ liệu ảnh để vẽ biểu đồ.')

---
## 📝 PHẦN B — TEXT EXTRACTION & VALIDATION

### B1. Đọc và trích xuất dữ liệu text từ CSV

In [ ]:
def extract_text_data(csv_path, text_col, label_col=None):
    """Đọc CSV và trích xuất cột text + label."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f'⚠️  File không tồn tại: {csv_path}')
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    print(f'📄 Đọc CSV thành công: {len(df)} dòng, {len(df.columns)} cột')
    print(f'   Các cột: {list(df.columns)}')

    if text_col not in df.columns:
        print(f'❌ Không tìm thấy cột "{text_col}" trong CSV!')
        print(f'   Các cột hiện có: {list(df.columns)}')
        return pd.DataFrame()

    keep_cols = [text_col]
    if label_col and label_col in df.columns:
        keep_cols.append(label_col)

    return df[keep_cols].copy()


df_text = extract_text_data(
    CONFIG['text_csv_path'],
    CONFIG['text_column'],
    CONFIG['label_column']
)

if not df_text.empty:
    display(df_text.head())

### B2. Validate dữ liệu text

In [ ]:
def validate_text_dataframe(df, text_col, config):
    """Thêm các cột validation vào DataFrame."""
    df = df.copy()
    col = text_col

    # --- Kiểm tra cơ bản ---
    df['is_null']       = df[col].isnull()
    df['text_clean']    = df[col].fillna('').astype(str).str.strip()
    df['char_length']   = df['text_clean'].str.len()
    df['word_count']    = df['text_clean'].str.split().str.len()
    df['is_empty']      = df['text_clean'] == ''

    # --- Kiểm tra độ dài ---
    df['too_short'] = df['char_length'] < config['min_text_length']
    df['too_long']  = df['char_length'] > config['max_text_length']

    # --- Kiểm tra duplicate ---
    df['is_duplicate'] = df['text_clean'].duplicated(keep=False)

    # --- Đánh giá tổng hợp ---
    df['is_valid'] = ~(
        df['is_null'] |
        df['is_empty'] |
        df['too_short'] |
        df['too_long']
    )

    return df


if not df_text.empty:
    df_text = validate_text_dataframe(df_text, CONFIG['text_column'], CONFIG)

    total   = len(df_text)
    valid   = df_text['is_valid'].sum()
    null    = df_text['is_null'].sum()
    empty   = df_text['is_empty'].sum()
    short   = df_text['too_short'].sum()
    long_   = df_text['too_long'].sum()
    dup     = df_text['is_duplicate'].sum()

    print(f'\n📊 KẾT QUẢ VALIDATION TEXT')
    print(f'   Tổng dòng    : {total}')
    print(f'   ✅ Hợp lệ    : {valid}  ({valid/total*100:.1f}%)')
    print(f'   🚫 Null      : {null}')
    print(f'   🚫 Trống     : {empty}')
    print(f'   📏 Quá ngắn  : {short}  (< {CONFIG["min_text_length"]} ký tự)')
    print(f'   📏 Quá dài   : {long_}  (> {CONFIG["max_text_length"]} ký tự)')
    print(f'   🔁 Duplicate : {dup}')
else:
    print('⚠️  Không có dữ liệu text để validate.')

### B3. Kiểm tra phân bố nhãn (Label Distribution)

In [ ]:
if not df_text.empty and CONFIG['label_column'] in df_text.columns:
    label_dist = df_text[CONFIG['label_column']].value_counts()
    print('📊 PHÂN BỐ NHÃN (LABEL DISTRIBUTION)')
    print(label_dist.to_string())

    # Cảnh báo imbalanced
    ratio = label_dist.max() / label_dist.min()
    if ratio > 5:
        print(f'\n⚠️  Dữ liệu MẤT CÂN BẰNG! Tỉ lệ max/min = {ratio:.1f}x')
    else:
        print(f'\n✅ Dữ liệu khá cân bằng. Tỉ lệ max/min = {ratio:.1f}x')
else:
    print('ℹ️  Không có cột label hoặc không có dữ liệu text.')

### B4. Visualize kết quả text

In [ ]:
if not df_text.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Text Validation Report', fontsize=14, fontweight='bold')

    # --- Biểu đồ 1: Valid vs Invalid ---
    counts = [df_text['is_valid'].sum(), (~df_text['is_valid']).sum()]
    axes[0].pie(counts, labels=['Valid', 'Invalid'],
                colors=['#4CAF50', '#F44336'],
                autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Tỉ lệ Valid / Invalid')

    # --- Biểu đồ 2: Phân bố độ dài ký tự ---
    valid_text = df_text[df_text['is_valid']]
    if not valid_text.empty:
        axes[1].hist(valid_text['char_length'], bins=30,
                     color='#9C27B0', edgecolor='white')
        axes[1].set_title('Phân bố độ dài văn bản (ký tự)')
        axes[1].set_xlabel('Số ký tự')
        axes[1].set_ylabel('Số lượng')

    # --- Biểu đồ 3: Label distribution ---
    if CONFIG['label_column'] in df_text.columns:
        label_dist = df_text[CONFIG['label_column']].value_counts()
        axes[2].bar(label_dist.index.astype(str), label_dist.values,
                    color='#00BCD4', edgecolor='white')
        axes[2].set_title('Phân bố nhãn (Label)')
        axes[2].set_xlabel('Label')
        axes[2].set_ylabel('Số lượng')
        plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')
    else:
        axes[2].text(0.5, 0.5, 'Không có cột label',
                     ha='center', va='center', transform=axes[2].transAxes)

    plt.tight_layout()
    report_path = f"{CONFIG['report_dir']}/text_validation.png"
    plt.savefig(report_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Đã lưu biểu đồ: {report_path}')
else:
    print('⚠️  Không có dữ liệu text để vẽ biểu đồ.')

---
## 📋 PHẦN C — XUẤT BÁO CÁO TỔNG HỢP

In [ ]:
report = {
    'image_summary': {},
    'text_summary' : {}
}

# --- Image summary ---
if not df_images.empty:
    report['image_summary'] = {
        'total'        : len(df_images),
        'valid'        : int(df_images['is_valid'].sum()),
        'invalid'      : int((~df_images['is_valid']).sum()),
        'corrupt'      : int(df_images['is_corrupt'].sum()),
        'formats'      : df_images['format'].value_counts().to_dict(),
        'avg_size_mb'  : round(df_images['size_mb'].mean(), 3)
    }

# --- Text summary ---
if not df_text.empty:
    report['text_summary'] = {
        'total'      : len(df_text),
        'valid'      : int(df_text['is_valid'].sum()),
        'null'       : int(df_text['is_null'].sum()),
        'empty'      : int(df_text['is_empty'].sum()),
        'too_short'  : int(df_text['too_short'].sum()),
        'too_long'   : int(df_text['too_long'].sum()),
        'duplicates' : int(df_text['is_duplicate'].sum()),
        'avg_char_length': round(df_text['char_length'].mean(), 1),
        'avg_word_count' : round(df_text['word_count'].mean(), 1)
    }

# --- Lưu JSON ---
report_json_path = f"{CONFIG['report_dir']}/validation_report.json"
with open(report_json_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# --- Lưu CSV lỗi ảnh ---
if not df_images.empty:
    invalid_images = df_images[~df_images['is_valid']]
    if not invalid_images.empty:
        invalid_images.to_csv(
            f"{CONFIG['report_dir']}/invalid_images.csv", index=False
        )

# --- Lưu CSV lỗi text ---
if not df_text.empty:
    invalid_texts = df_text[~df_text['is_valid']]
    if not invalid_texts.empty:
        invalid_texts.to_csv(
            f"{CONFIG['report_dir']}/invalid_texts.csv", index=False
        )

print('✅ Báo cáo đã được lưu vào thư mục:', CONFIG['report_dir'])
print('   📄 validation_report.json')
print('   📄 invalid_images.csv      (nếu có lỗi)')
print('   📄 invalid_texts.csv       (nếu có lỗi)')
print('   🖼️  image_validation.png')
print('   🖼️  text_validation.png')
print()
print('📊 TÓM TẮT:')
print(json.dumps(report, ensure_ascii=False, indent=2))